# Imports

In [ ]:
!pip install mne boto3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import mne
import gc

# Dataset creations: Phased Manner

First create 6 master dataframes which hold the data for the trials

create them in a phased manner to avoid RAM overload... trial wise

In [ ]:
# --- 0. Imports ---
import pandas as pd
import numpy as np
import mne
import boto3
import tempfile
from google.colab import userdata

# --- 1. S3 Configuration (from Colab Secrets) ---
s3 = boto3.client(
    's3',
    aws_access_key_id     = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name           = 'ap-south-1'
)

S3_BUCKET       = 'amzn-eeg-bucket'
S3_INPUT_PREFIX = 'korean/Cluster_1_epochs_data'

# --- 2. Other Configuration ---
subjects_list = [f"{i:02d}" for i in range(1, 56)]
runs = {
    'train': ['1', '2'],
    'test':  ['1','2', '3', '4']
}
output_dir = '/content/drive/MyDrive/P300_Expanded/final/korean'
os.makedirs(output_dir, exist_ok=True)
tmp_dir = tempfile.mkdtemp()

# --- 3. Process runs one by one ---
for run_type, run_nums in runs.items():
    for run_num in run_nums:
        run_key = f"{run_type}_{run_num}"
        output_file = os.path.join(output_dir, f'{run_key}.csv')
        header_written = False
        print(f"Processing {run_key} ...")

        for subject in subjects_list:
            run_name = f"s{subject}_{run_type}_{run_num}"
            tmp_mat  = os.path.join(tmp_dir, f"{run_name}_epoch_data.mat")
            tmp_txt  = os.path.join(tmp_dir, f"{run_name}_true_labels.txt")

            try:
                s3.download_file(S3_BUCKET, f"{S3_INPUT_PREFIX}/{run_name}_epoch_data.mat", tmp_mat)
                s3.download_file(S3_BUCKET, f"{S3_INPUT_PREFIX}/{run_name}_true_labels.txt", tmp_txt)

                epochs        = mne.io.read_epochs_eeglab(tmp_mat, verbose=False)
                eeg_data      = epochs.get_data()
                n_epochs, n_channels, n_times = eeg_data.shape
                channel_names = epochs.ch_names

                # Labels are newline-separated
                with open(tmp_txt, 'r') as f:
                    labels = [int(line.strip()) for line in f if line.strip()]

                if n_epochs != len(labels):
                    print(f"Epoch/label mismatch for {run_name}, skipping.")
                    continue
                if n_times != 512:
                    print(f"Unexpected timepoints {n_times} for {run_name}, skipping.")
                    continue

            except Exception as e:
                print(f"Error loading {run_name}: {e}")
                continue
            finally:
                for path in [tmp_mat, tmp_txt]:
                    if os.path.exists(path): os.remove(path)

            for i in range(n_epochs):
                rows = []
                for j in range(n_channels):
                    row = {
                        'subject_id':   subject,
                        'run_type':     run_type,
                        'run_id':       run_num,
                        'epoch_num':    i,
                        'epoch_type':   'target' if labels[i] == 1 else 'non-target',
                        'channel_name': channel_names[j],
                        **{f"t_{k}": eeg_data[i, j, k] for k in range(n_times)}
                    }
                    rows.append(row)

                df = pd.DataFrame(rows)
                df.to_csv(output_file, mode='a', header=not header_written, index=False)
                header_written = True

            del eeg_data, epochs

print("All CSVs created.")

Processing train_1 ...


/tmp/ipykernel_8992/2642317823.py:48: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_mat, verbose=False)
/tmp/ipykernel_8992/2642317823.py:48: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_mat, verbose=False)
/tmp/ipykernel_8992/2642317823.py:48: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_mat, verbose=False)
/tmp/ipykernel_8992/2642317823.py:48: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_mat, verbose=False)
/tmp/ipykernel_8992/2642317823.py:48: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will b

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Doing for giga


[subj01 sess01 train]


Traceback (most recent call last):
  File "/tmp/ipykernel_1684/1121164680.py", line 55, in <cell line: 0>
    s3.download_file(S3_BUCKET, f"{S3_INPUT_PREFIX}/{run_name}.fdt", tmp_fdt)
  File "/usr/local/lib/python3.12/dist-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/boto3/s3/inject.py", line 223, in download_file
    return transfer.download_file(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/boto3/s3/transfer.py", line 484, in download_file
    future.result()
  File "/usr/local/lib/python3.12/dist-packages/s3transfer/futures.py", line 111, in result
    return self._coordinator.result()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/s3transfer/futures.py", line 287, in result
    raise self._exception
  File "/usr/local/lib/python3.12/dist-packages/s3transfer/tasks.py", line 272, in _main


NameError: name 'n_epochs' is not defined

In [ ]:
response = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_INPUT_PREFIX
)

for obj in response.get('Contents', []):
    print(obj['Key'])

giga_preprocessed/
giga_preprocessed/clean_epoched_final_1.fdt
giga_preprocessed/clean_epoched_final_1.set
giga_preprocessed/clean_epoched_subj01_sess01_EEG_ERP_test.set
giga_preprocessed/clean_epoched_subj01_sess01_EEG_ERP_train.set
giga_preprocessed/clean_epoched_subj01_sess02_EEG_ERP_test.set
giga_preprocessed/clean_epoched_subj01_sess02_EEG_ERP_train.set
giga_preprocessed/clean_epoched_subj02_sess01_EEG_ERP_test.set
giga_preprocessed/clean_epoched_subj02_sess01_EEG_ERP_train.set
giga_preprocessed/clean_epoched_subj02_sess02_EEG_ERP_test.set
giga_preprocessed/clean_epoched_subj02_sess02_EEG_ERP_train.set
giga_preprocessed/clean_epoched_subj03_sess01_EEG_ERP_test.fdt
giga_preprocessed/clean_epoched_subj03_sess01_EEG_ERP_test.set
giga_preprocessed/clean_epoched_subj03_sess01_EEG_ERP_train.fdt
giga_preprocessed/clean_epoched_subj03_sess01_EEG_ERP_train.set
giga_preprocessed/clean_epoched_subj03_sess02_EEG_ERP_test.fdt
giga_preprocessed/clean_epoched_subj03_sess02_EEG_ERP_test.set
giga_

In [ ]:
# --- 0. Imports ---
import pandas as pd
import numpy as np
import os
import mne
import boto3
import tempfile
import io
from google.colab import userdata

# --- 1. S3 Configuration ---
s3 = boto3.client(
    's3',
    aws_access_key_id     = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name           = 'ap-south-1'
)

S3_BUCKET        = 'amzn-eeg-bucket'
S3_INPUT_PREFIX  = 'giga_preprocessed'
S3_OUTPUT_PREFIX = 'giga_preprocessed/parquet_data'

# --- 2. Configuration ---
subjects_list = [f"{i:02d}" for i in range(1, 55)]   # 01 to 54
sessions_list = ['01', '02']
split_list    = ['train', 'test']
N_TIMES       = 410

tmp_dir = tempfile.mkdtemp()

# --- 3. Helper: check if S3 key already exists (safe resume) ---
def s3_key_exists(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        return False

# --- 4. Process all combinations ---
for sess in sessions_list:
    for split in split_list:
        for subj in subjects_list:
            run_name = f"clean_epoched_subj{subj}_sess{sess}_EEG_ERP_{split}"
            out_key  = f"{S3_OUTPUT_PREFIX}/sess{sess}/{split}/subject_{subj}.parquet"
            tmp_set  = os.path.join(tmp_dir, f"{run_name}.set")
            #tmp_fdt  = os.path.join(tmp_dir, f"{run_name}.fdt")

            # Safe resume
            if s3_key_exists(S3_BUCKET, out_key):
                print(f"  [subj{subj} sess{sess} {split}] already exists, skipping.")
                continue

            try:
                s3.download_file(S3_BUCKET, f"{S3_INPUT_PREFIX}/{run_name}.set", tmp_set)
                #s3.download_file(S3_BUCKET, f"{S3_INPUT_PREFIX}/{run_name}.fdt", tmp_fdt)

                epochs        = mne.io.read_epochs_eeglab(tmp_set, verbose=False)
                eeg_data      = epochs.get_data()  # (n_epochs, n_channels, n_times)
                n_epochs, n_channels, n_times = eeg_data.shape
                channel_names = epochs.ch_names

                # Recover original MATLAB labels via inverted event_id
                code_to_label = {v: int(k) for k, v in epochs.event_id.items()}
                labels        = [code_to_label[code] for code in epochs.events[:, 2]]

                if n_epochs != len(labels):
                    print(f"  [subj{subj} sess{sess} {split}] Epoch/label mismatch, skipping.")
                    continue
                if n_times != N_TIMES:
                    print(f"  [subj{subj} sess{sess} {split}] Unexpected timepoints {n_times}, skipping.")
                    continue

            except Exception as e:
                print(f"  [subj{subj} sess{sess} {split}] Error loading: {e}")
                continue
            finally:
                for path in [tmp_set]:
                    if os.path.exists(path): os.remove(path)

            # --- 5. Build DataFrame ---
            rows = []
            for i in range(n_epochs):
                for j in range(n_channels):
                    rows.append({
                        'subject_id':   subj,
                        'session_id':   sess,
                        'split':        split,
                        'epoch_num':    i,
                        'epoch_type':   'target' if labels[i] == 2 else 'non-target',
                        'channel_name': channel_names[j],
                        **{f"t_{k}": eeg_data[i, j, k] for k in range(n_times)}
                    })

            del eeg_data, epochs

            # --- 6. Upload parquet directly to S3 ---
            df  = pd.DataFrame(rows)
            buf = io.BytesIO()
            df.to_parquet(buf, index=False, engine='pyarrow', compression='snappy')
            buf.seek(0)
            s3.put_object(Bucket=S3_BUCKET, Key=out_key, Body=buf.getvalue())
            print(f"  [subj{subj} sess{sess} {split}] uploaded → s3://{S3_BUCKET}/{out_key}  ({len(df)} rows)")

            del df, rows, buf
            gc.collect()

print("\nAll parquet files uploaded.")

  [subj01 sess01 train] already exists, skipping.
  [subj02 sess01 train] already exists, skipping.
  [subj03 sess01 train] already exists, skipping.
  [subj04 sess01 train] already exists, skipping.
  [subj05 sess01 train] already exists, skipping.
  [subj06 sess01 train] already exists, skipping.
  [subj07 sess01 train] already exists, skipping.
  [subj08 sess01 train] already exists, skipping.
  [subj09 sess01 train] already exists, skipping.
  [subj10 sess01 train] already exists, skipping.
  [subj11 sess01 train] already exists, skipping.
  [subj12 sess01 train] already exists, skipping.
  [subj13 sess01 train] already exists, skipping.
  [subj14 sess01 train] already exists, skipping.
  [subj15 sess01 train] already exists, skipping.
  [subj16 sess01 train] already exists, skipping.
  [subj17 sess01 train] already exists, skipping.
  [subj18 sess01 train] already exists, skipping.
  [subj19 sess01 train] already exists, skipping.
  [subj20 sess01 train] already exists, skipping.


# Adaptive P300 data extraction

In [ ]:
# --- 0. Imports ---
import pandas as pd
import numpy as np
import os
import mne
import boto3
import tempfile
import io
import scipy.io as sio
from google.colab import userdata

# --- 1. S3 Configuration ---
s3 = boto3.client(
    's3',
    aws_access_key_id     = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name           = 'ap-south-1'
)

S3_BUCKET        = 'amzn-eeg-bucket'
S3_PROCESSED     = 'adaptiveP300/processed'
S3_OUTPUT_PREFIX = 'adaptiveP300/parquet_data'

# --- 2. Configuration ---
subjects_list = [f"{i:02d}" for i in range(1, 48)]
file_types    = [
    'calibration-signals',
    'eval-raw',
    'training-run-1-raw',
    'training-run-2-raw',
    'training-run-3-raw',
    'training-run-4-raw',
    'training-run-5-raw',
    'post-training-raw',
]
N_TIMES = 500

tmp_dir = tempfile.mkdtemp()

# --- 3. Helpers ---
def s3_key_exists(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except s3.exceptions.ClientError as e:
        if e.response['Error']['Code'] == '404':
            return False
        raise

def extract_labels(epoch_struct):
    """Exact Python equivalent of MATLAB time-zero label extraction."""
    labels = []
    for e in epoch_struct:
        latencies  = np.array(e.eventlatency).flatten()
        eventtypes = np.array(e.eventtype).flatten()
        zero_idx   = np.where(latencies == 0)[0]
        if len(zero_idx) == 0:
            labels.append('unknown')  # flag for inspection
            continue
        evt_type = eventtypes[zero_idx[0]]
        labels.append('target' if evt_type == 'B1(1)' else 'non-target')
    return labels

# --- 4. Process all combinations ---
for file_type in file_types:
    for subj in subjects_list:
        base_name = f"Subject-{subj}_{file_type}_clean"
        out_key   = f"{S3_OUTPUT_PREFIX}/{file_type}/subject_{subj}.parquet"
        tmp_set   = os.path.join(tmp_dir, f"{base_name}.set")
        tmp_fdt   = os.path.join(tmp_dir, f"{base_name}.fdt")

        if s3_key_exists(S3_BUCKET, out_key):
            print(f"  [subj{subj} {file_type}] already exists, skipping.")
            continue

        try:
            s3.download_file(S3_BUCKET, f"{S3_PROCESSED}/{base_name}.set", tmp_set)
            s3.download_file(S3_BUCKET, f"{S3_PROCESSED}/{base_name}.fdt", tmp_fdt)

            # MNE for EEG data
            epochs        = mne.io.read_epochs_eeglab(tmp_set, verbose=False)
            eeg_data      = epochs.get_data()
            n_epochs, n_channels, n_times = eeg_data.shape
            channel_names = epochs.ch_names

            # scipy for exact time-zero label extraction (mirrors MATLAB loop)
            raw_mat       = sio.loadmat(tmp_set, squeeze_me=True, struct_as_record=False)
            epoch_struct  = raw_mat['epoch']
            labels        = extract_labels(epoch_struct)

            if n_epochs != len(labels):
                print(f"  [subj{subj} {file_type}] Epoch/label mismatch ({n_epochs} vs {len(labels)}), skipping.")
                continue
            if n_times != N_TIMES:
                print(f"  [subj{subj} {file_type}] Unexpected timepoints {n_times}, skipping.")
                continue

            # Warn if any unknown labels
            n_unknown = labels.count('unknown')
            if n_unknown > 0:
                print(f"  [subj{subj} {file_type}] WARNING: {n_unknown} epochs with no time-zero event.")

        except Exception as e:
            print(f"  [subj{subj} {file_type}] Error loading: {e}")
            continue
        finally:
            for path in [tmp_set, tmp_fdt]:
                if os.path.exists(path): os.remove(path)

        # --- 5. Build DataFrame ---
        rows = []
        for i in range(n_epochs):
            for j in range(n_channels):
                rows.append({
                    'subject_id':   subj,
                    'file_type':    file_type,
                    'epoch_num':    i,
                    'epoch_type':   labels[i],
                    'channel_name': channel_names[j],
                    **{f"t_{k}": eeg_data[i, j, k] for k in range(n_times)}
                })

        del eeg_data, epochs, raw_mat, epoch_struct

        # --- 6. Upload parquet directly to S3 ---
        df  = pd.DataFrame(rows)
        buf = io.BytesIO()
        df.to_parquet(buf, index=False, engine='pyarrow', compression='snappy')
        buf.seek(0)
        s3.put_object(Bucket=S3_BUCKET, Key=out_key, Body=buf.getvalue())
        print(f"  [subj{subj} {file_type}] uploaded → s3://{S3_BUCKET}/{out_key}  ({len(df)} rows)")

        del df, rows, buf
        gc.collect()

print("\nAll parquet files uploaded.")

  [subj01 calibration-signals] already exists, skipping.
  [subj02 calibration-signals] already exists, skipping.
  [subj03 calibration-signals] already exists, skipping.
  [subj04 calibration-signals] already exists, skipping.
  [subj05 calibration-signals] already exists, skipping.
  [subj06 calibration-signals] already exists, skipping.
  [subj07 calibration-signals] already exists, skipping.
  [subj08 calibration-signals] already exists, skipping.
  [subj09 calibration-signals] already exists, skipping.
  [subj10 calibration-signals] already exists, skipping.
  [subj11 calibration-signals] already exists, skipping.
  [subj12 calibration-signals] already exists, skipping.
  [subj13 calibration-signals] already exists, skipping.
  [subj14 calibration-signals] already exists, skipping.
  [subj15 calibration-signals] already exists, skipping.
  [subj16 calibration-signals] already exists, skipping.
  [subj17 calibration-signals] already exists, skipping.
  [subj18 calibration-signals] 

/tmp/ipykernel_2160/3569009020.py:81: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_set, verbose=False)


  [subj31 calibration-signals] uploaded → s3://amzn-eeg-bucket/adaptiveP300/parquet_data/calibration-signals/subject_31.parquet  (36608 rows)


/tmp/ipykernel_2160/3569009020.py:81: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_set, verbose=False)


  [subj32 calibration-signals] uploaded → s3://amzn-eeg-bucket/adaptiveP300/parquet_data/calibration-signals/subject_32.parquet  (36608 rows)


...

  [subj45 post-training-raw] uploaded → s3://amzn-eeg-bucket/adaptiveP300/parquet_data/post-training-raw/subject_45.parquet  (22880 rows)


/tmp/ipykernel_2160/3569009020.py:81: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_set, verbose=False)


  [subj46 post-training-raw] uploaded → s3://amzn-eeg-bucket/adaptiveP300/parquet_data/post-training-raw/subject_46.parquet  (22880 rows)


/tmp/ipykernel_2160/3569009020.py:81: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs        = mne.io.read_epochs_eeglab(tmp_set, verbose=False)


  [subj47 post-training-raw] uploaded → s3://amzn-eeg-bucket/adaptiveP300/parquet_data/post-training-raw/subject_47.parquet  (22880 rows)

All parquet files uploaded.


In [ ]:
# Quick probe — run this as a standalone cell
import mne
import boto3
import tempfile
import os
from google.colab import userdata

s3 = boto3.client(
    's3',
    aws_access_key_id     = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name           = 'ap-south-1'
)

S3_BUCKET    = 'amzn-eeg-bucket'
S3_PROCESSED = 'adaptiveP300/processed'
tmp_dir      = tempfile.mkdtemp()

# Probe Subject-01 calibration-signals
base_name = 'Subject-01_calibration-signals_clean'
tmp_set   = os.path.join(tmp_dir, f'{base_name}.set')
tmp_fdt   = os.path.join(tmp_dir, f'{base_name}.fdt')

s3.download_file(S3_BUCKET, f'{S3_PROCESSED}/{base_name}.set', tmp_set)
s3.download_file(S3_BUCKET, f'{S3_PROCESSED}/{base_name}.fdt', tmp_fdt)

epochs = mne.io.read_epochs_eeglab(tmp_set, verbose=False)

print("=== event_id ===")
print(epochs.event_id)

print("\n=== events (first 10) ===")
print(epochs.events[:10])

print("\n=== metadata ===")
print(epochs.metadata)

print("\n=== shape ===")
print(epochs.get_data().shape)

for i in range(5):
    event_code    = epochs.events[i, 2]
    event_sample  = epochs.events[i, 0]
    event_id_str  = {v: k for k, v in epochs.event_id.items()}[event_code]
    first_token   = event_id_str.split('/')[0]

    print(f"=== Epoch {i} ===")
    print(f"  Sample latency : {event_sample}")
    print(f"  MNE code       : {event_code}")
    print(f"  Full event_id  : {event_id_str}")
    print(f"  First token    : {first_token}")
    print(f"  Assigned label : {'target' if first_token == 'B1(1)' else 'non-target'}")
    print()

# Also check overall target ratio
code_to_event = {v: k for k, v in epochs.event_id.items()}
labels = [code_to_event[code].split('/')[0] for code in epochs.events[:, 2]]
from collections import Counter
print("=== Label distribution ===")
print(Counter(labels))


=== event_id ===
{'B2(2)/B2(2)/B2(2)/B2(2)/B1(1)': 1, 'B2(2)/B2(2)/B2(2)/B2(2)/B1(1)/B1(1)': 2, 'B2(2)/B2(2)/B2(2)/B1(1)/B1(1)/B2(2)': 3, 'B2(2)/B2(2)/B1(1)/B1(1)/B2(2)/B2(2)': 4, 'B2(2)/B1(1)/B1(1)/B2(2)/B2(2)/B2(2)': 5, 'B1(1)/B1(1)/B2(2)/B2(2)/B2(2)/B2(2)': 6, 'B1(1)/B2(2)/B2(2)/B2(2)/B2(2)/B2(2)': 7, 'B2(2)/B2(2)/B2(2)/B2(2)/B2(2)/B2(2)': 8, 'B2(2)/B2(2)/B2(2)/B2(2)/B2(2)': 9, 'B2(2)/B2(2)/B2(2)/B1(1)/B2(2)': 10, 'B2(2)/B2(2)/B2(2)/B1(1)/B2(2)/B2(2)': 11, 'B2(2)/B2(2)/B1(1)/B2(2)/B2(2)/B2(2)': 12, 'B2(2)/B1(1)/B2(2)/B2(2)/B2(2)/B2(2)': 13, 'B1(1)/B2(2)/B2(2)/B2(2)/B2(2)/B1(1)': 14, 'B2(2)/B2(2)/B2(2)/B2(2)/B1(1)/B2(2)': 15, 'B2(2)/B1(1)/B2(2)/B2(2)/B2(2)': 16, 'B1(1)/B2(2)/B2(2)/B2(2)/B2(2)': 17, 'B2(2)/B2(2)/B1(1)/B2(2)/B2(2)/B1(1)': 18, 'B2(2)/B1(1)/B2(2)/B2(2)/B1(1)/B2(2)': 19, 'B1(1)/B2(2)/B2(2)/B1(1)/B2(2)/B2(2)': 20, 'B2(2)/B2(2)/B1(1)/B2(2)/B2(2)': 21, 'B2(2)/B2(2)/B2(2)/B2(2)/B2(2)/B1(1)': 22, 'B2(2)/B2(2)/B1(1)/B1(1)/B2(2)': 23, 'B2(2)/B2(2)/B2(2)/B1(1)/B2(2)/B1(1)': 24, '

/tmp/ipykernel_2160/3921693823.py:27: RuntimeWarning: At least one epoch has multiple events. Only the latency of the first event will be retained.
  epochs = mne.io.read_epochs_eeglab(tmp_set, verbose=False)


In [ ]:
# Inspect raw EEGLAB epoch structure for first 3 epochs
import scipy.io as sio

# Load the .set file directly as a mat file to see raw epoch structure
raw = sio.loadmat(tmp_set, squeeze_me=True, struct_as_record=False)
EEG = raw['EEG']

print("=== EEG.epoch[0] eventlatency ===")
print(EEG.epoch[0].eventlatency)

print("\n=== EEG.epoch[0] eventtype ===")
print(EEG.epoch[0].eventtype)

print("\n=== EEG.epoch[1] eventlatency ===")
print(EEG.epoch[1].eventlatency)

print("\n=== EEG.epoch[1] eventtype ===")
print(EEG.epoch[1].eventtype)


KeyError: 'EEG'

In [ ]:

epoch = raw['epoch']

print(type(epoch))
print(epoch.shape)

# Inspect first 3 epochs
for i in range(20):
    print(f"\n=== Epoch {i} ===")
    print(f"  eventlatency : {epoch[i].eventlatency}")
    print(f"  eventtype    : {epoch[i].eventtype}")


<class 'numpy.ndarray'>
(1144,)

=== Epoch 0 ===
  eventlatency : [0 178 350 522 694]
  eventtype    : ['B2(2)' 'B2(2)' 'B2(2)' 'B2(2)' 'B1(1)']

=== Epoch 1 ===
  eventlatency : [-178 0 172 344 516 688]
  eventtype    : ['B2(2)' 'B2(2)' 'B2(2)' 'B2(2)' 'B1(1)' 'B1(1)']

=== Epoch 2 ===
  eventlatency : [-172 0 172 344 516 688]
  eventtype    : ['B2(2)' 'B2(2)' 'B2(2)' 'B1(1)' 'B1(1)' 'B2(2)']

=== Epoch 3 ===
  eventlatency : [-172 0 172 344 516 688]
  eventtype    : ['B2(2)' 'B2(2)' 'B1(1)' 'B1(1)' 'B2(2)' 'B2(2)']

=== Epoch 4 ===
  eventlatency : [-172 0 172 344 516 688]
  eventtype    : ['B2(2)' 'B1(1)' 'B1(1)' 'B2(2)' 'B2(2)' 'B2(2)']

=== Epoch 5 ===
  eventlatency : [-172 0 172 344 516 686]
  eventtype    : ['B1(1)' 'B1(1)' 'B2(2)' 'B2(2)' 'B2(2)' 'B2(2)']

=== Epoch 6 ===
  eventlatency : [-172 0 172 344 514 688]
  eventtype    : ['B1(1)' 'B2(2)' 'B2(2)' 'B2(2)' 'B2(2)' 'B2(2)']

=== Epoch 7 ===
  eventlatency : [-172 0 172 342 516 686]
  eventtype    : ['B2(2)' 'B2(2)' 'B2(2)